# Macro Dashboard: Visualizing the U.S. Economy as a System

Most macroeconomic data is consumed one number at a time — a headline inflation print, a monthly jobs report, a rate decision. Read in isolation, these figures hide the thing that matters most: they are part of a coupled system that moves together, with lags.

This notebook pulls four core U.S. indicators from the [FRED API](https://fred.stlouisfed.org/) and visualizes how they relate to one another over time:

| Indicator | FRED series | Frequency | Role in the system |
|---|---|---|---|
| Real GDP growth | `A191RL1Q225SBEA` | Quarterly | Baseline measure of expansion vs. contraction |
| CPI inflation | `CPIAUCSL` (index → YoY %) | Monthly | Price-stability half of the Fed's dual mandate |
| Unemployment rate | `UNRATE` | Monthly | Labor-market half of the mandate |
| Federal funds rate | `FEDFUNDS` | Monthly | The Fed's policy lever — the link that ties the other three together |

I also pull `USREC` (NBER recession indicator) to shade recessions in every chart, since recessions are the regime shifts around which most of these relationships bend.

**How to read this notebook.** Each visualization is preceded by the economic reasoning behind it — why these series should co-move, what lag to expect, and where the textbook story tends to break down — and followed by notes on what the data actually shows.

## 1. Setup
I first got a free FREDAPI key ([here](https://fred.stlouisfed.org/docs/api/api_key.html)) to pull data. The key is read from the `FRED_API_KEY` environment variable so it never appears in the notebook or the repo.

In [14]:
import sys
print(sys.executable)

c:\Users\tasso\AppData\Local\Programs\Python\Python313\python.exe


In [15]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from fredapi import Fred
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_KEY = os.environ.get("FRED_API_KEY")
assert API_KEY, "Set the FRED_API_KEY environment variable (or put it in a .env file)."

fred = Fred(api_key=API_KEY)

## 2. Data pipeline

### 2.1 Pulling the raw series

The four indicators arrive at different frequencies and in different units: GDP growth is already a quarterly, seasonally-adjusted annualized rate; CPI is a monthly *index level* that has to be converted into an inflation rate; unemployment and the funds rate are monthly percentages that can be used as-is.

**Time window (decision point):** the pipeline defaults to `1990-01-01` onward — long enough to cover the Volcker aftermath's disinflation regime, the 1990s expansion, the dot-com and 2008 recessions, the ZIRP decade, and the 2021–23 inflation episode, while keeping the charts readable. Starting in the 1960s–70s adds the Great Inflation (useful for the Phillips-curve section) at the cost of visual clutter. Change `START_DATE` below to experiment.

In [ ]:
START_DATE = "1990-01-01"   # <-- decision point: widen to e.g. "1960-01-01" for the Great Inflation era

SERIES = {
    "gdp_growth": "A191RL1Q225SBEA",  # Real GDP growth, % chg SAAR, quarterly
    "cpi_index":  "CPIAUCSL",         # CPI, index level, monthly
    "unemp":      "UNRATE",           # Unemployment rate, %, monthly
    "fedfunds":   "FEDFUNDS",         # Effective federal funds rate, %, monthly
    "recession":  "USREC",            # NBER recession indicator (0/1), monthly
}

raw = {name: fred.get_series(code, observation_start=START_DATE)
       for name, code in SERIES.items()}

for name, s in raw.items():
    print(f"{name:>10}: {len(s):>4} obs | {s.index.min().date()} → {s.index.max().date()}")

gdp_growth:  145 obs | 1990-01-01 → 2026-01-01
 cpi_index:  437 obs | 1990-01-01 → 2026-05-01
     unemp:  438 obs | 1990-01-01 → 2026-06-01
  fedfunds:  438 obs | 1990-01-01 → 2026-06-01
 recession:  438 obs | 1990-01-01 → 2026-06-01


### 2.2 Transformation and alignment

Two things have to happen before the series are comparable:

1. **CPI index → inflation rate.** The headline "inflation" number is the year-over-year percent change in the CPI index, so I compute `pct_change(12)` on the monthly index. YoY (rather than month-over-month annualized) trades responsiveness for smoothness — it's the measure the Fed's communication and the public discussion actually revolve around.
2. **Frequency alignment.** GDP is quarterly; everything else is monthly. Rather than throwing away monthly detail, I keep a **monthly master frame** and forward-fill the quarterly GDP figure across the months of its quarter. This treats the quarterly print as the best available estimate of growth *during* that quarter, which is the standard choice for visualization (for formal econometrics you'd be more careful — e.g., aggregating monthly series to quarterly instead).

In [17]:
# CPI index -> year-over-year inflation rate
inflation = raw["cpi_index"].pct_change(12) * 100

# Monthly master frame
df = pd.DataFrame({
    "inflation": inflation,
    "unemp":     raw["unemp"],
    "fedfunds":  raw["fedfunds"],
    "recession": raw["recession"],
})

# Align quarterly GDP growth onto the monthly index (forward-fill within quarter)
gdp = raw["gdp_growth"].copy()
gdp.index = gdp.index.to_period("Q").to_timestamp()      # normalize to quarter start
df["gdp_growth"] = gdp.reindex(df.index, method="ffill")

df = df.dropna(subset=["inflation"])                     # first 12 months lost to YoY calc
df.index.name = "date"

df.tail()

C:\Users\tasso\AppData\Local\Temp\ipykernel_9176\4048626668.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  inflation = raw["cpi_index"].pct_change(12) * 100


,inflation,unemp,fedfunds,recession,gdp_growth
date,,,,,
2026-01-01,2.391201,4.3,3.64,0.0,2.1
2026-02-01,2.434004,4.4,3.64,0.0,2.1
2026-03-01,3.285958,4.3,3.64,0.0,2.1
2026-04-01,3.779246,4.3,3.64,0.0,2.1
2026-05-01,4.166615,4.3,3.63,0.0,2.1


In [18]:
# Helper: recession spans for shading, derived from USREC
def recession_spans(rec: pd.Series):
    rec = rec.fillna(0).astype(int)
    changes = rec.diff().fillna(rec.iloc[0])
    starts = rec.index[changes == 1].tolist()
    ends   = rec.index[changes == -1].tolist()
    if rec.iloc[-1] == 1:                                # recession ongoing at sample end
        ends.append(rec.index[-1])
    return list(zip(starts, ends))

RECESSIONS = recession_spans(df["recession"])

def add_recession_shading(fig, spans=RECESSIONS, **kwargs):
    for start, end in spans:
        fig.add_vrect(x0=start, x1=end, fillcolor="gray", opacity=0.18,
                      line_width=0, layer="below", **kwargs)
    return fig

RECESSIONS

[(Timestamp('1991-01-01 00:00:00'), Timestamp('1991-04-01 00:00:00')),
 (Timestamp('2001-04-01 00:00:00'), Timestamp('2001-12-01 00:00:00')),
 (Timestamp('2008-01-01 00:00:00'), Timestamp('2009-07-01 00:00:00')),
 (Timestamp('2020-03-01 00:00:00'), Timestamp('2020-05-01 00:00:00'))]

## 3. Visualization 1 — The system at a glance

**Economic reasoning.** Before looking at any pairwise relationship, it's worth seeing all four series on a shared time axis with recessions marked. The stylized sequence to look for around each recession is:

1. Inflation rises → 2. the Fed raises the funds rate → 3. growth slows and, with a lag, unemployment rises → 4. the Fed cuts, → 5. recovery, and eventually the cycle restarts.

The chart below stacks the four series in linked panels (shared x-axis, so zooming one zooms all). The gray bands are NBER recessions. The interesting question isn't whether the sequence appears — it's *how the timing and magnitude differ across cycles*: 2008 vs. the 2022 tightening, for example, look very different in the unemployment panel.

In [19]:
fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    subplot_titles=("Real GDP growth (% SAAR)", "CPI inflation (% YoY)",
                    "Unemployment rate (%)", "Federal funds rate (%)"),
)

panels = [("gdp_growth", "#1f77b4"), ("inflation", "#d62728"),
          ("unemp", "#2ca02c"), ("fedfunds", "#9467bd")]

for i, (col, color) in enumerate(panels, start=1):
    fig.add_trace(go.Scatter(x=df.index, y=df[col], name=col,
                             line=dict(color=color, width=1.6)), row=i, col=1)

add_recession_shading(fig)
fig.add_hline(y=0, line_dash="dot", line_color="gray", row=1, col=1)

fig.update_layout(height=850, showlegend=False, hovermode="x unified",
                  title="The U.S. macro system, {} – present (gray = NBER recessions)".format(START_DATE[:4]),
                  margin=dict(t=90))
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

**What the data shows (fill in after running).** Draft observations to verify and expand in my own words:

- Every recession in the window is preceded by a tightening cycle in the bottom panel — but not every tightening cycle is followed by a recession (candidate counterexamples: 1994–95, and so far 2022–23).
- Unemployment is the clearest *lagging* indicator: it typically peaks well after the recession has formally ended.
- The 2020 recession is the outlier for the whole notebook — a shutdown, not a policy-induced downturn — and it will distort any correlation computed across it.

## 4. Visualization 2 — Policy transmission: the funds rate leads, unemployment follows

**Economic reasoning.** Monetary policy famously operates with "long and variable lags" (Friedman). A rate hike doesn't destroy jobs the month it happens: it raises borrowing costs, which slows investment and hiring over the following quarters. The conventional estimate is that the peak effect on the real economy arrives **roughly 12–24 months** after the policy change.

Two ways to see this:

1. **Overlay with a shift.** Plot the funds rate shifted forward *k* months against the unemployment rate. If policy transmits with lag *k*, the shifted funds rate should line up with subsequent unemployment movements.
2. **Cross-correlation profile.** Compute the correlation between the *change* in the funds rate and the *change* in unemployment at every lag from 0 to 36 months. Using changes (first differences) rather than levels avoids the spurious correlation that plagues trending, highly persistent series — a classic econometrics trap.

In [ ]:
LAG_MONTHS = 18   # try 12, 18, 24

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df.index, y=df["fedfunds"].shift(LAG_MONTHS),
                         name=f"Fed funds rate (shifted +{LAG_MONTHS}m)",
                         line=dict(color="#9467bd")), secondary_y=False)
fig.add_trace(go.Scatter(x=df.index, y=df["unemp"], name="Unemployment rate",
                         line=dict(color="#2ca02c")), secondary_y=True)

add_recession_shading(fig)
fig.update_layout(title=f"Policy leads, the labor market follows: funds rate shifted {LAG_MONTHS} months vs. unemployment",
                  hovermode="x unified", height=500,
                  legend=dict(orientation="h", y=1.1))
fig.update_yaxes(title_text="Funds rate (%)", secondary_y=False)
fig.update_yaxes(title_text="Unemployment (%)", secondary_y=True)
fig.show()

In [ ]:
# Cross-correlation of monthly CHANGES: Δfedfunds(t) vs Δunemp(t + k)
d_ff = df["fedfunds"].diff()
d_un = df["unemp"].diff()

lags = range(0, 37)
xcorr = [d_ff.corr(d_un.shift(-k)) for k in lags]   # shift(-k): unemployment k months LATER

fig = go.Figure(go.Bar(x=list(lags), y=xcorr, marker_color="#9467bd"))
fig.update_layout(title="Corr( Δ funds rate today , Δ unemployment k months later )",
                  xaxis_title="Lag k (months)", yaxis_title="Correlation",
                  height=420)
fig.show()

best = int(np.nanargmax(xcorr))
print(f"Correlation peaks at lag ≈ {best} months (ρ = {xcorr[best]:.3f})")

**What the data shows (fill in after running).** Things I expect and want to check honestly:

- The raw cross-correlations will be **small** — single-month changes are noisy, and lots of other shocks move unemployment. The interesting result is the *shape* of the profile (near zero at short lags, rising over 12–24 months), not a large ρ.
- The 2020 COVID spike is a huge outlier in Δunemployment; it's worth re-running the profile excluding 2020 to see how much it drives the result.
- Correlation at a lag is **not** causal identification. Establishing that policy *causes* the unemployment response requires dealing with the Fed's own reaction to the economy (endogeneity) — the honest framing here is "the timing pattern is consistent with the transmission story," nothing stronger.

## 5. Visualization 3 — The Phillips curve, and where it breaks

**Economic reasoning.** The Phillips curve is the classic claimed tradeoff: low unemployment ⇒ tight labor markets ⇒ wage pressure ⇒ inflation. If it held cleanly, a scatter of unemployment against inflation would slope downward.

It famously does not hold cleanly. The relationship shifted after the 1970s (expectations became anchored), looked nearly **flat** through the 2010s — unemployment fell to 50-year lows with no inflation to show for it — and then the 2021–23 episode produced high inflation driven substantially by supply factors, not labor-market tightness alone.

Plotting the scatter **colored by decade** turns a "broken" relationship into the actual finding: the curve isn't one curve, it's a family of regime-specific curves that shift with expectations and supply conditions.

In [ ]:
scatter_df = df.dropna(subset=["unemp", "inflation"]).copy()
scatter_df["decade"] = (scatter_df.index.year // 10 * 10).astype(str) + "s"

fig = go.Figure()
for decade, g in scatter_df.groupby("decade"):
    fig.add_trace(go.Scatter(
        x=g["unemp"], y=g["inflation"], mode="markers", name=decade,
        marker=dict(size=6, opacity=0.65),
        customdata=g.index.strftime("%b %Y"),
        hovertemplate="%{customdata}<br>Unemp: %{x:.1f}%<br>Inflation: %{y:.1f}%<extra></extra>",
    ))

fig.update_layout(title="Phillips curve scatter by decade: one relationship, or several?",
                  xaxis_title="Unemployment rate (%)",
                  yaxis_title="CPI inflation (% YoY)",
                  height=550)
fig.show()

**What the data shows (fill in after running).** Draft observations:

- Within a single decade the points often trace a recognizable downward-sloping cloud; **across** decades the clouds sit at different heights — the visual signature of shifting inflation expectations.
- The 2010s cluster should be nearly horizontal: unemployment ranges widely while inflation barely moves — the "flat Phillips curve" that dominated pre-COVID policy debate.
- The 2020s points will be the strangest: 2020 (high unemployment, low inflation) and 2022 (low unemployment, high inflation) are nearly opposite corners within three years, which is hard to square with a labor-market-only story and points to supply shocks.

## 6. Visualization 4 — The policy stance: nominal rates, inflation, and the real rate

**Economic reasoning.** The level of the funds rate by itself doesn't tell you whether policy is tight or loose — a 5% funds rate is restrictive when inflation is 2% and *stimulative* when inflation is 9%. The crude but informative measure is the **real funds rate**: nominal funds rate minus current YoY inflation (an *ex post* real rate; a proper *ex ante* measure would subtract expected inflation).

The chart overlays the funds rate and inflation, then plots the real rate below. Episodes where inflation runs **above** the funds rate — a negative real rate — are periods where the Fed is, in real terms, behind the curve or deliberately accommodative. The 2021–22 gap between the two lines is one of the widest in the sample and is exactly the visual version of the "the Fed was late" argument.

In [ ]:
df["real_rate"] = df["fedfunds"] - df["inflation"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    row_heights=[0.6, 0.4],
                    subplot_titles=("Federal funds rate vs. CPI inflation",
                                    "Real funds rate (funds rate − YoY inflation)"))

fig.add_trace(go.Scatter(x=df.index, y=df["fedfunds"], name="Fed funds rate",
                         line=dict(color="#9467bd")), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["inflation"], name="CPI inflation (YoY)",
                         line=dict(color="#d62728")), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df["real_rate"], name="Real funds rate",
                         line=dict(color="#17becf"),
                         fill="tozeroy", fillcolor="rgba(23,190,207,0.15)"), row=2, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="gray", row=2, col=1)

add_recession_shading(fig)
fig.update_layout(title="How tight is policy, really?", height=650,
                  hovermode="x unified", legend=dict(orientation="h", y=1.08))
fig.show()

**What the data shows (fill in after running).** Draft observations:

- The real rate is strongly **procyclical in policy terms**: deeply negative after 2008 and again in 2021–22, positive during deliberate tightening episodes (2000, 2006–07, 2023–24).
- The 2021–22 stretch should show the most negative real rate in the window — the quantitative core of the "behind the curve" debate.
- Caveat worth stating in the final write-up: subtracting *realized* inflation overstates how loose policy looked in real time, because policymakers act on *expected* inflation. A refinement would swap in a market- or survey-based expectations series (e.g., FRED `T5YIE` or `MICH`).

## 7. Conclusions and extensions

**The system view (to finalize after all charts are run):**

- The four indicators do move as a coupled system, but the coupling is loose, lagged, and regime-dependent — the textbook sequence (inflation → hikes → slack → cuts) is visible in the timing, while the magnitudes differ enormously across cycles.
- The most robust single pattern is the *lag structure*: policy moves show up in the labor market with a delay on the order of a year or more.
- The most instructive failures are the flat 2010s Phillips curve and the 2021–23 inflation, both of which show why single-equation textbook relationships need regime and supply-side context.

**Planned extensions:**

- [ ] Streamlit dashboard: date-range slider, lag slider for the transmission chart, indicator toggles → deployed with a live link
- [ ] Add inflation-expectations series (`T5YIE`, `MICH`) and recompute the real-rate chart *ex ante*
- [ ] Rolling correlation windows to show relationship instability over time
- [ ] A simple Taylor-rule benchmark plotted against the actual funds rate

---

*Data: Federal Reserve Economic Data (FRED), Federal Reserve Bank of St. Louis. This notebook is an exploratory visualization project; nothing here is causal identification or investment advice.*